## 1. Install the system packages needed for the build

This cell installs the C++ compiler, `make`, FFmpeg for MP4 encoding, and WebP development headers for the preview image.


In [1]:
%%bash
set -e
apt-get update
apt-get install -y g++ make ffmpeg libwebp-dev


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,534 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Create the project folder

This cell sets up the working directory and requests up to 12 render threads. The renderer itself clamps that request to the actual number of OpenMP threads available on the runtime.

In [2]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/raytrace_submission")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

AVAILABLE_THREADS = os.cpu_count() or 2
RENDER_THREADS = min(12, AVAILABLE_THREADS)

print("Project directory:", PROJECT_DIR)
print("Available CPU threads:", AVAILABLE_THREADS)
print("Requested render threads:", 12)
print("Renderer will actually use up to:", RENDER_THREADS)


Project directory: /content/raytrace_submission
Available CPU threads: 12
Requested render threads: 12
Renderer will actually use up to: 12


## Write `interval.h`

This snippet writes the interval helper. It handles numeric ranges and clamping, which are used for ray hit limits and gamma-safe color output.

In [3]:
%%writefile /content/raytrace_submission/interval.h
#ifndef INTERVAL_H
#define INTERVAL_H
//==============================================================================================
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

class interval {
  public:
    double min, max;

    interval() : min(+infinity), max(-infinity) {} // Default interval is empty

    interval(double min, double max) : min(min), max(max) {}

    double size() const {
        return max - min;
    }

    bool contains(double x) const {
        return min <= x && x <= max;
    }

    bool surrounds(double x) const {
        return min < x && x < max;
    }

    double clamp(double x) const {
        if (x < min) return min;
        if (x > max) return max;
        return x;
    }

    static const interval empty, universe;
};

const interval interval::empty    = interval(+infinity, -infinity);
const interval interval::universe = interval(-infinity, +infinity);


#endif


Writing /content/raytrace_submission/interval.h


## Write `color.h`

This snippet writes the color utilities. It defines the color alias and the gamma conversion helper used when pixels are written to the final image.

In [4]:
%%writefile /content/raytrace_submission/color.h
#ifndef COLOR_H
#define COLOR_H
//==============================================================================================
// Originally written in 2020 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "interval.h"
#include "vec3.h"

using color = vec3;


inline double linear_to_gamma(double linear_component)
{
    if (linear_component > 0)
        return std::sqrt(linear_component);

    return 0;
}


void write_color(std::ostream& out, const color& pixel_color) {
    auto r = pixel_color.x();
    auto g = pixel_color.y();
    auto b = pixel_color.z();

    // Apply a linear to gamma transform for gamma 2
    r = linear_to_gamma(r);
    g = linear_to_gamma(g);
    b = linear_to_gamma(b);

    // Translate the [0,1] component values to the byte range [0,255].
    static const interval intensity(0.000, 0.999);
    int rbyte = int(256 * intensity.clamp(r));
    int gbyte = int(256 * intensity.clamp(g));
    int bbyte = int(256 * intensity.clamp(b));

    // Write out the pixel color components.
    out << rbyte << ' ' << gbyte << ' ' << bbyte << '\n';
#if 0
    if (rbyte != (pixel & 0xFF))
        std::cout << "red ";
    if (gbyte != ((pixel >> 8) & 0xFF))
        std::cout << "green ";
    if (bbyte != ((pixel >> 16) & 0xFF))
        std::cout << "blue";
#endif
}


#endif


Writing /content/raytrace_submission/color.h


## Write `vec3.h`

This snippet writes the core 3D vector math type. It powers points, colors, directions, dot products, cross products, reflection, and refraction.

In [5]:
%%writefile /content/raytrace_submission/vec3.h
#pragma once

#include <iostream>
#include <cmath>
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

class vec3 {
  public:
    double e[3];

    vec3() : e{0,0,0} {}
    vec3(double e0, double e1, double e2) : e{e0, e1, e2} {}

    double x() const { return e[0]; }
    double y() const { return e[1]; }
    double z() const { return e[2]; }

    vec3 operator-() const { return vec3(-e[0], -e[1], -e[2]); }
    double operator[](int i) const { return e[i]; }
    double& operator[](int i) { return e[i]; }

    vec3& operator+=(const vec3& v) {
        e[0] += v.e[0];
        e[1] += v.e[1];
        e[2] += v.e[2];
        return *this;
    }

    vec3& operator*=(double t) {
        e[0] *= t;
        e[1] *= t;
        e[2] *= t;
        return *this;
    }

    vec3& operator/=(double t) {
        return *this *= 1/t;
    }

    double length() const {
        return std::sqrt(length_squared());
    }

    double length_squared() const {
        return e[0]*e[0] + e[1]*e[1] + e[2]*e[2];
    }

    bool near_zero() const {
        // Return true if the vector is close to zero in all dimensions.
        auto s = 1e-8;
        return (std::fabs(e[0]) < s) && (std::fabs(e[1]) < s) && (std::fabs(e[2]) < s);
    }

    static vec3 random() {
        return vec3(random_double(), random_double(), random_double());
    }

    static vec3 random(double min, double max) {
        return vec3(random_double(min,max), random_double(min,max), random_double(min,max));
    }
};

// point3 is just an alias for vec3, but useful for geometric clarity in the code.
using point3 = vec3;


// Vector Utility Functions

inline vec3 operator+(const vec3& u, const vec3& v) {
    return vec3(u.e[0] + v.e[0], u.e[1] + v.e[1], u.e[2] + v.e[2]);
}

inline vec3 operator-(const vec3& u, const vec3& v) {
    return vec3(u.e[0] - v.e[0], u.e[1] - v.e[1], u.e[2] - v.e[2]);
}

inline vec3 operator*(const vec3& u, const vec3& v) {
    return vec3(u.e[0] * v.e[0], u.e[1] * v.e[1], u.e[2] * v.e[2]);
}

inline vec3 operator*(double t, const vec3& v) {
    return vec3(t*v.e[0], t*v.e[1], t*v.e[2]);
}

inline vec3 operator*(const vec3& v, double t) {
    return t * v;
}

inline vec3 operator/(const vec3& v, double t) {
    return (1/t) * v;
}

inline double dot(const vec3& u, const vec3& v) {
    return u.e[0] * v.e[0]
         + u.e[1] * v.e[1]
         + u.e[2] * v.e[2];
}

inline vec3 cross(const vec3& u, const vec3& v) {
    return vec3(u.e[1] * v.e[2] - u.e[2] * v.e[1],
                u.e[2] * v.e[0] - u.e[0] * v.e[2],
                u.e[0] * v.e[1] - u.e[1] * v.e[0]);
}

inline vec3 unit_vector(const vec3& v) {
    return v / v.length();
}

inline vec3 random_in_unit_disk() {
    while (true) {
        auto p = vec3(random_double(-1,1), random_double(-1,1), 0);
        if (p.length_squared() < 1)
            return p;
    }
}

inline vec3 random_unit_vector() {
    while (true) {
        auto p = vec3::random(-1,1);
        auto lensq = p.length_squared();
        if (1e-160 < lensq && lensq <= 1.0)
            return p / sqrt(lensq);
    }
}

inline vec3 random_on_hemisphere(const vec3& normal) {
    vec3 on_unit_sphere = random_unit_vector();
    if (dot(on_unit_sphere, normal) > 0.0) // In the same hemisphere as the normal
        return on_unit_sphere;
    else
        return -on_unit_sphere;
}

inline vec3 reflect(const vec3& v, const vec3& n) {
    return v - 2*dot(v,n)*n;
}

inline vec3 refract(const vec3& uv, const vec3& n, double etai_over_etat) {
    auto cos_theta = std::fmin(dot(-uv, n), 1.0);
    vec3 r_out_perp =  etai_over_etat * (uv + cos_theta*n);
    vec3 r_out_parallel = -std::sqrt(std::fabs(1.0 - r_out_perp.length_squared())) * n;
    return r_out_perp + r_out_parallel;
}

inline std::ostream& operator<<(std::ostream& out, const vec3& v) {
    return out << v.e[0] << ' ' << v.e[1] << ' ' << v.e[2];
}

inline std::istream& operator>>(std::istream& in, vec3& v) {
    return in >> v.e[0] >> v.e[1] >> v.e[2];
}



Writing /content/raytrace_submission/vec3.h


## Write `ray.h`

This snippet writes the ray type. A ray stores an origin and direction and is the fundamental object traced through the scene.

In [6]:
%%writefile /content/raytrace_submission/ray.h
#pragma once
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "vec3.h"


class ray {
  public:
    ray() {}

    ray(const point3& origin, const vec3& direction) : orig(origin), dir(direction) {}

    const point3& origin() const  { return orig; }
    const vec3& direction() const { return dir; }

    point3 at(double t) const {
        return orig + t*dir;
    }

  private:
    point3 orig;
    vec3 dir;
};


Writing /content/raytrace_submission/ray.h


## Write `rtweekend.h`

This snippet writes the shared runtime header. It now uses a **thread-local random engine** for thread safety and also exposes a **deterministic reseed helper**. The renderer reseeds the sampler per pixel and per frame so the noise pattern is stable even with OpenMP and dynamic scheduling.

In [7]:
%%writefile /content/raytrace_submission/rtweekend.h
        #pragma once
        //==============================================================================================
        // To the extent possible under law, the author(s) have dedicated all copyright and related and
        // neighboring rights to this software to the public domain worldwide. This software is
        // distributed without any warranty.
        //
        // You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
        // along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
        //==============================================================================================

        #include <cmath>
        #include <cstdlib>
        #include <iostream>
        #include <limits>
        #include <memory>
        #include <cstdint>
        #include <random>
        #include <thread>
        #include <functional>

        // C++ Std Usings

        using std::make_shared;
        using std::shared_ptr;


        // Constants

        const double infinity = std::numeric_limits<double>::infinity();
        const double pi = 3.1415926535897932385;


        // Utility Functions

        inline double degrees_to_radians(double degrees) {
            return degrees * (pi / 180.0);
        }

        // SplitMix64-style bit mixer. It turns nearby integer seeds into well-separated
        // generator states so neighboring pixels do not share correlated sample patterns.
        inline std::uint64_t mix_u64(std::uint64_t x) {
            x += 0x9e3779b97f4a7c15ULL;
            x = (x ^ (x >> 30)) * 0xbf58476d1ce4e5b9ULL;
            x = (x ^ (x >> 27)) * 0x94d049bb133111ebULL;
            return x ^ (x >> 31);
        }

        inline std::mt19937_64& global_rng() {
            static thread_local std::mt19937_64 generator(0x1234abcd5678ef90ULL);
            return generator;
        }

        inline void seed_rng(std::uint64_t seed) {
            global_rng().seed(mix_u64(seed));
        }

        inline double random_double() {
            static thread_local std::uniform_real_distribution<double> distribution(0.0, 1.0);
            return distribution(global_rng());
        }

        inline double random_double(double min, double max) {
            return min + (max - min) * random_double();
        }


        // Common Headers

        #include "color.h"
        #include "interval.h"
        #include "ray.h"
        #include "vec3.h"


Writing /content/raytrace_submission/rtweekend.h


## Write `hittable.h`

This snippet writes the base hit interface and the hit record. The hit record stores the intersection point, normal, material, and now the sphere UV coordinates.

In [8]:
%%writefile /content/raytrace_submission/hittable.h
#pragma once
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include <memory>
#include "ray.h"
#include "aabb.h"
class material;

class hit_record {
  public:
    point3 p;
    vec3 normal;
    shared_ptr<material> mat;
    double t;
    double u;
    double v;
    bool front_face;

    void set_face_normal(const ray& r, const vec3& outward_normal) {
        // Sets the hit record normal vector.
        // NOTE: the parameter `outward_normal` is assumed to have unit length.

        front_face = dot(r.direction(), outward_normal) < 0;
        normal = front_face ? outward_normal : -outward_normal;
    }
};


class hittable {
  public:
    virtual ~hittable() = default;

    virtual bool hit(const ray& r, interval ray_t, hit_record& rec) const = 0;
    virtual aabb bounding_box() const = 0;
};


Writing /content/raytrace_submission/hittable.h


## Write `hittable_list.h`

This snippet writes the container that stores many hittable objects. It loops over the scene and keeps the closest valid intersection.

In [9]:
%%writefile /content/raytrace_submission/hittable_list.h
#pragma once
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "hittable.h"

#include <vector>


class hittable_list : public hittable {
  public:
    std::vector<shared_ptr<hittable>> objects;
    aabb bbox;

    hittable_list() {}
    hittable_list(shared_ptr<hittable> object) { add(object); }

    void clear() {
        objects.clear();
        bbox = aabb();
    }

    void add(shared_ptr<hittable> object) {
        if (objects.empty()) bbox = object->bounding_box();
        else bbox = surrounding_box(bbox, object->bounding_box());
        objects.push_back(object);
    }

    bool hit(const ray& r, interval ray_t, hit_record& rec) const override {
        hit_record temp_rec;
        bool hit_anything = false;
        auto closest_so_far = ray_t.max;

        for (const auto& object : objects) {
            if (object->hit(r, interval(ray_t.min, closest_so_far), temp_rec)) {
                hit_anything = true;
                closest_so_far = temp_rec.t;
                rec = temp_rec;
            }
        }

        return hit_anything;
    }

    aabb bounding_box() const override { return bbox; }
};


Writing /content/raytrace_submission/hittable_list.h


## Write `aabb.h`

This snippet writes the axis-aligned bounding box helper. BVH nodes use it to reject large groups of objects quickly.

In [10]:
%%writefile /content/raytrace_submission/aabb.h
#pragma once

#include "interval.h"
#include "ray.h"

class aabb {
  public:
    interval x, y, z;

    aabb() {}
    aabb(const interval& x, const interval& y, const interval& z) : x(x), y(y), z(z) {}
    aabb(const point3& a, const point3& b)
        : x(interval(std::fmin(a.x(), b.x()), std::fmax(a.x(), b.x()))),
          y(interval(std::fmin(a.y(), b.y()), std::fmax(a.y(), b.y()))),
          z(interval(std::fmin(a.z(), b.z()), std::fmax(a.z(), b.z()))) {}

    const interval& axis_interval(int axis) const {
        if (axis == 0) return x;
        if (axis == 1) return y;
        return z;
    }

    int longest_axis() const {
        if (x.size() > y.size()) {
            return x.size() > z.size() ? 0 : 2;
        }
        return y.size() > z.size() ? 1 : 2;
    }

    bool hit(const ray& r, interval ray_t) const {
        for (int axis = 0; axis < 3; ++axis) {
            const interval& ax = axis_interval(axis);
            const double inv_d = 1.0 / r.direction()[axis];
            double t0 = (ax.min - r.origin()[axis]) * inv_d;
            double t1 = (ax.max - r.origin()[axis]) * inv_d;

            if (inv_d < 0.0) {
                std::swap(t0, t1);
            }

            if (t0 > ray_t.min) ray_t.min = t0;
            if (t1 < ray_t.max) ray_t.max = t1;
            if (ray_t.max <= ray_t.min) {
                return false;
            }
        }
        return true;
    }
};

inline aabb surrounding_box(const aabb& box0, const aabb& box1) {
    interval x(
        std::fmin(box0.x.min, box1.x.min),
        std::fmax(box0.x.max, box1.x.max)
    );
    interval y(
        std::fmin(box0.y.min, box1.y.min),
        std::fmax(box0.y.max, box1.y.max)
    );
    interval z(
        std::fmin(box0.z.min, box1.z.min),
        std::fmax(box0.z.max, box1.z.max)
    );
    return aabb(x, y, z);
}


Writing /content/raytrace_submission/aabb.h


## Write `bvh.h`

This snippet writes the bounding volume hierarchy. The comparator now breaks ties consistently so nearby spheres do not reorder unpredictably when their bounding boxes are nearly identical.

In [11]:
%%writefile /content/raytrace_submission/bvh.h
        #pragma once

        #include <algorithm>
        #include <vector>
        #include "hittable.h"

        class bvh_node : public hittable {
          public:
            bvh_node() {}

            bvh_node(std::vector<shared_ptr<hittable>>& src_objects)
                : bvh_node(src_objects, 0, src_objects.size()) {}

            bvh_node(std::vector<shared_ptr<hittable>>& src_objects, size_t start, size_t end) {
                auto objects = src_objects;

                aabb centroid_bounds;
                bool first_box = true;
                for (size_t i = start; i < end; ++i) {
                    aabb box = objects[i]->bounding_box();
                    if (first_box) {
                        centroid_bounds = box;
                        first_box = false;
                    } else {
                        centroid_bounds = surrounding_box(centroid_bounds, box);
                    }
                }

                const int axis = centroid_bounds.longest_axis();
                auto comparator = [axis](const shared_ptr<hittable>& a, const shared_ptr<hittable>& b) {
                    const interval& ai = a->bounding_box().axis_interval(axis);
                    const interval& bi = b->bounding_box().axis_interval(axis);
                    constexpr double eps = 1e-12;

                    if (ai.min < bi.min - eps) return true;
                    if (ai.min > bi.min + eps) return false;
                    if (ai.max < bi.max - eps) return true;
                    if (ai.max > bi.max + eps) return false;

                    // Stable tie-breaker when bounds are effectively identical.
                    return a.get() < b.get();
                };

                size_t object_span = end - start;

                if (object_span == 1) {
                    left = right = objects[start];
                } else if (object_span == 2) {
                    if (comparator(objects[start], objects[start + 1])) {
                        left = objects[start];
                        right = objects[start + 1];
                    } else {
                        left = objects[start + 1];
                        right = objects[start];
                    }
                } else {
                    std::sort(objects.begin() + start, objects.begin() + end, comparator);
                    size_t mid = start + object_span / 2;
                    left = make_shared<bvh_node>(objects, start, mid);
                    right = make_shared<bvh_node>(objects, mid, end);
                }

                bbox = surrounding_box(left->bounding_box(), right->bounding_box());
            }

            bool hit(const ray& r, interval ray_t, hit_record& rec) const override {
                if (!bbox.hit(r, ray_t)) return false;

                bool hit_left = left->hit(r, ray_t, rec);
                bool hit_right = right->hit(r, interval(ray_t.min, hit_left ? rec.t : ray_t.max), rec);

                return hit_left || hit_right;
            }

            aabb bounding_box() const override { return bbox; }

          private:
            shared_ptr<hittable> left;
            shared_ptr<hittable> right;
            aabb bbox;
        };


Writing /content/raytrace_submission/bvh.h


## Write `material.h`

This snippet writes the material system. The important addition here is the spinning UV material, which uses sphere UV coordinates plus animation time so the balls show visible rotation.

In [12]:
%%writefile /content/raytrace_submission/material.h
#pragma once
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "hittable.h"

extern double g_anim_time_seconds;

inline double wrap01(double x) {
    x = std::fmod(x, 1.0);
    return x < 0.0 ? x + 1.0 : x;
}

inline color lerp_color(const color& a, const color& b, double t) {
    return (1.0 - t) * a + t * b;
}


class material {
  public:
    virtual ~material() = default;

    virtual color emitted() const {
        return color(0, 0, 0);
    }

    virtual bool scatter(
        const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered
    ) const {
        return false;
    }
};


class lambertian : public material {
  public:
    lambertian(const color& albedo) : albedo(albedo) {}

    bool scatter(const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered)
    const override {
        auto scatter_direction = rec.normal + random_unit_vector();

        // Catch degenerate scatter direction
        if (scatter_direction.near_zero())
            scatter_direction = rec.normal;

        scattered = ray(rec.p, scatter_direction);
        attenuation = albedo;
        return true;
    }

  private:
    color albedo;
};


class spinning_uv_lambertian : public material {
  public:
    spinning_uv_lambertian(
        const color& color_a,
        const color& color_b,
        double spin_cycles_per_second,
        double lon_frequency = 10.0,
        double lat_frequency = 6.0,
        double edge_mix = 0.18)
      : color_a(color_a),
        color_b(color_b),
        spin_cycles_per_second(spin_cycles_per_second),
        lon_frequency(lon_frequency),
        lat_frequency(lat_frequency),
        edge_mix(edge_mix) {}

    bool scatter(const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered)
    const override {
        auto scatter_direction = rec.normal + random_unit_vector();
        if (scatter_direction.near_zero())
            scatter_direction = rec.normal;

        const double u = wrap01(rec.u + g_anim_time_seconds * spin_cycles_per_second);
        const double v = wrap01(rec.v);

        const double stripes = 0.5 + 0.5 * std::sin(2.0 * pi * lon_frequency * u);
        const double bands   = 0.5 + 0.5 * std::sin(2.0 * pi * lat_frequency * v);
        const double ring    = 0.5 + 0.5 * std::sin(2.0 * pi * (lon_frequency * 0.5 * u + lat_frequency * 0.35 * v));
        const double pattern = 0.55 * stripes + 0.30 * bands + 0.15 * ring;

        const double pole_glow = std::pow(std::fabs(2.0 * v - 1.0), 3.0);
        const double mixv = std::min(1.0, std::max(0.0, pattern * (1.0 - edge_mix) + edge_mix * pole_glow));

        attenuation = lerp_color(color_a, color_b, mixv);
        scattered = ray(rec.p, scatter_direction);
        return true;
    }

  private:
    color color_a;
    color color_b;
    double spin_cycles_per_second;
    double lon_frequency;
    double lat_frequency;
    double edge_mix;
};


class metal : public material {
  public:
    metal(const color& albedo, double fuzz) : albedo(albedo), fuzz(fuzz < 1 ? fuzz : 1) {}

    bool scatter(const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered)
    const override {
        vec3 reflected = reflect(r_in.direction(), rec.normal);
        reflected = unit_vector(reflected) + (fuzz * random_unit_vector());
        scattered = ray(rec.p, reflected);
        attenuation = albedo;
        return (dot(scattered.direction(), rec.normal) > 0);
    }

  private:
    color albedo;
    double fuzz;
};


class dielectric : public material {
  public:
    dielectric(double refraction_index) : refraction_index(refraction_index) {}

    bool scatter(const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered)
    const override {
        attenuation = color(1.0, 1.0, 1.0);
        double ri = rec.front_face ? (1.0/refraction_index) : refraction_index;

        vec3 unit_direction = unit_vector(r_in.direction());
        double cos_theta = std::fmin(dot(-unit_direction, rec.normal), 1.0);
        double sin_theta = std::sqrt(1.0 - cos_theta*cos_theta);

        bool cannot_refract = ri * sin_theta > 1.0;
        vec3 direction;

        if (cannot_refract || reflectance(cos_theta, ri) > random_double())
            direction = reflect(unit_direction, rec.normal);
        else
            direction = refract(unit_direction, rec.normal, ri);

        scattered = ray(rec.p, direction);
        return true;
    }

  private:
    // Refractive index in vacuum or air, or the ratio of the material's refractive index over
    // the refractive index of the enclosing media
    double refraction_index;

    static double reflectance(double cosine, double refraction_index) {
        // Use Schlick's approximation for reflectance.
        auto r0 = (1 - refraction_index) / (1 + refraction_index);
        r0 = r0*r0;
        return r0 + (1-r0)*std::pow((1 - cosine),5);
    }
};


class diffuse_light : public material {
  public:
    diffuse_light(const color& emit) : emit(emit) {}

    color emitted() const override {
        return emit;
    }

    bool scatter(const ray& r_in, const hit_record& rec, color& attenuation, ray& scattered)
    const override {
        return false;
    }

  private:
    color emit;
};


Writing /content/raytrace_submission/material.h


## Write `sphere.h`

This snippet writes the sphere primitive. The key addition is UV mapping, which converts a hit point on the sphere into texture coordinates so surface motion becomes visible.

In [13]:
%%writefile /content/raytrace_submission/sphere.h
#ifndef SPHERE_H
#define SPHERE_H
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "hittable.h"

class sphere : public hittable {
  public:
    sphere(const point3& center, double radius, shared_ptr<material> mat)
      : center(center), radius(std::fmax(0,radius)), mat(mat) {
        vec3 radius_vec(this->radius, this->radius, this->radius);
        bbox = aabb(center - radius_vec, center + radius_vec);
      }

    static void get_sphere_uv(const point3& p, double& u, double& v) {
        const auto theta = std::acos(-p.y());
        const auto phi = std::atan2(-p.z(), p.x()) + pi;
        u = phi / (2*pi);
        v = theta / pi;
    }

    bool hit(const ray& r, interval ray_t, hit_record& rec) const override {
        vec3 oc = center - r.origin();
        auto a = r.direction().length_squared();
        auto h = dot(r.direction(), oc);
        auto c = oc.length_squared() - radius*radius;

        auto discriminant = h*h - a*c;
        if (discriminant < 0)
            return false;

        auto sqrtd = std::sqrt(discriminant);

        // Find the nearest root that lies in the acceptable range.
        auto root = (h - sqrtd) / a;
        if (!ray_t.surrounds(root)) {
            root = (h + sqrtd) / a;
            if (!ray_t.surrounds(root))
                return false;
        }

        rec.t = root;
        rec.p = r.at(rec.t);
        vec3 outward_normal = (rec.p - center) / radius;
        get_sphere_uv(outward_normal, rec.u, rec.v);
        rec.set_face_normal(r, outward_normal);
        rec.mat = mat;

        return true;
    }

    aabb bounding_box() const override { return bbox; }

  private:
    point3 center;
    double radius;
    shared_ptr<material> mat;
    aabb bbox;
};


#endif


Writing /content/raytrace_submission/sphere.h


## Write `camera.h`

This snippet writes the camera and renderer. It keeps **dynamic OpenMP scheduling**, honors the requested thread count, and now reseeds the random sampler **per pixel and per frame** so the animation remains temporally stable instead of flickering from thread scheduling changes.

In [14]:
%%writefile /content/raytrace_submission/camera.h
#pragma once
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================

#include "hittable.h"
#include "material.h"
#include <chrono>
#include <fstream>
#include <iomanip>
#include <atomic>
#ifdef _OPENMP
#include <omp.h>
#endif

class camera {
  public:
    double aspect_ratio;          // Ratio of image width over height
    int    image_width;           // Rendered image width in pixel count
    int    image_height;         // Rendered image height
    int    samples_per_pixel;     // Count of random samples for each pixel
    int    max_depth;             // Maximum number of ray bounces into scene

    double vfov;                  // Vertical view angle (field of view)
    point3 lookfrom;              // Point camera is looking from
    point3 lookat;                // Point camera is looking at
    vec3   vup;                  // Camera-relative "up" direction

    double defocus_angle;        // Variation angle of rays through each pixel
    double focus_dist;           // Distance from camera lookfrom point to plane of perfect focus
    bool   use_space_shader;     // Use iterative black-space shader with explicit light sphere
    point3 light_center;         // Light sphere center for space shader
    double light_radius;         // Light sphere radius for space shader
    color  light_color;          // Emitted color when ray hits light sphere
    int    render_threads;       // Requested OpenMP thread count (0 = automatic)

    camera() {
        aspect_ratio = 1.0;
        image_width = 100;
        samples_per_pixel = 10;
        max_depth = 10;
        vfov = 90;
        lookfrom = point3(0,0,0);
        lookat = point3(0,0,-1);
        vup = vec3(0,1,0);
        defocus_angle = 0;
        focus_dist = 10;
        use_space_shader = false;
        light_center = point3(0, 8, 0);
        light_radius = 1.0;
        light_color = color(1, 1, 1);
        render_threads = 0;
    }


    inline uint32_t conv_color(float c) {
        if (c < 0)
          return 0;
        if (c > 0.999) c = 0.999;
        return uint32_t(255.999 * sqrt(c));
    }

    inline uint32_t conv_color(const color& pixel_color) {
        auto r = linear_to_gamma(pixel_color.x());
        auto g = linear_to_gamma(pixel_color.y());
        auto b = linear_to_gamma(pixel_color.z());

        // Translate the [0,1] component values to the byte range [0,255].
        static const interval intensity(0.000, 0.999);
        int rbyte = int(256 * intensity.clamp(r));
        int gbyte = int(256 * intensity.clamp(g));
        int bbyte = int(256 * intensity.clamp(b));

        return 0xFF000000 | (bbyte << 16) | (gbyte << 8) | rbyte;
    }



    void render(const hittable& world, uint32_t* pixels, const std::string& filename, int frame) {
        initialize();
        auto t0 = std::chrono::high_resolution_clock::now();
        std::cout << "Scanlines remaining:\n";

#ifdef _OPENMP
        int requested_threads = render_threads > 0 ? std::min(render_threads, omp_get_max_threads()) : omp_get_max_threads();
#else
        int requested_threads = 1;
#endif
        std::atomic<int> rows_done(0);

#pragma omp parallel for schedule(dynamic, 1) num_threads(requested_threads)
        for (int j = 0; j < image_height; ++j) {
            for (int i = 0; i < image_width; ++i) {
                const std::uint64_t pixel_seed =
                    mix_u64((static_cast<std::uint64_t>(frame + 1) << 40) ^
                            (static_cast<std::uint64_t>(j + 1) << 20) ^
                             static_cast<std::uint64_t>(i + 1));
                seed_rng(pixel_seed);

                color pixel_color(0,0,0);
                for (int sample = 0; sample < samples_per_pixel; ++sample) {
                    ray r = get_ray(i, j);
                    if (use_space_shader) {
                        pixel_color += ray_color_space(r, max_depth, world);
                    } else {
                        pixel_color += ray_color(r, max_depth, world);
                    }
                }
                pixels[j*image_width + i] = conv_color(pixel_samples_scale * pixel_color);
            }

            int completed = ++rows_done;
            if (completed % 10 == 0 || completed == image_height) {
#pragma omp critical(progress_output)
                {
                    std::cout << '\r' << (image_height - completed) << ' ' << std::flush;
                }
            }
        }
        std::cout << "\n";

        if (frame == 0) {
            std::ofstream f2(filename + ".ppm");
            f2 << "P3\n" << image_width << ' ' << image_height << "\n255\n";
            for (int j = 0; j < image_height; ++j) {
                for (int i = 0; i < image_width; ++i) {
                    const uint32_t p = pixels[j*image_width + i];
                    const int r = p & 0xFF;
                    const int g = (p >> 8) & 0xFF;
                    const int b = (p >> 16) & 0xFF;
                    f2 << r << ' ' << g << ' ' << b << "\n";
                }
            }
        }

        auto t1 = std::chrono::high_resolution_clock::now();
        std::cerr << "Elapsed: " << std::chrono::duration_cast<std::chrono::milliseconds>(t1 - t0).count() << "ms\n";
    }

  private:
    double pixel_samples_scale;  // Color scale factor for a sum of pixel samples
    point3 center;               // Camera center
    point3 pixel00_loc;          // Location of pixel 0, 0
    vec3   pixel_delta_u;        // Offset to pixel to the right
    vec3   pixel_delta_v;        // Offset to pixel below
    vec3   u, v, w;              // Camera frame basis vectors
    vec3   defocus_disk_u;       // Defocus disk horizontal radius
    vec3   defocus_disk_v;       // Defocus disk vertical radius

    void initialize() {

        pixel_samples_scale = 1.0 / samples_per_pixel;

        center = lookfrom;

        // Determine viewport dimensions.
        auto theta = degrees_to_radians(vfov);
        auto h = std::tan(theta/2);
        auto viewport_height = 2 * h * focus_dist;
        auto viewport_width = viewport_height * (double(image_width)/image_height);

        // Calculate the u,v,w unit basis vectors for the camera coordinate frame.
        w = unit_vector(lookfrom - lookat);
        u = unit_vector(cross(vup, w));
        v = cross(w, u);

        // Calculate the vectors across the horizontal and down the vertical viewport edges.
        vec3 viewport_u = viewport_width * u;    // Vector across viewport horizontal edge
        vec3 viewport_v = viewport_height * -v;  // Vector down viewport vertical edge

        // Calculate the horizontal and vertical delta vectors from pixel to pixel.
        pixel_delta_u = viewport_u / image_width;
        pixel_delta_v = viewport_v / image_height;

        // Calculate the location of the upper left pixel.
        auto viewport_upper_left = center - (focus_dist * w) - viewport_u/2 - viewport_v/2;
        pixel00_loc = viewport_upper_left + 0.5 * (pixel_delta_u + pixel_delta_v);

        // Calculate the camera defocus disk basis vectors.
        auto defocus_radius = focus_dist * std::tan(degrees_to_radians(defocus_angle / 2));
        defocus_disk_u = u * defocus_radius;
        defocus_disk_v = v * defocus_radius;
    }

    ray get_ray(int i, int j) const {
        // Construct a camera ray originating from the defocus disk and directed at a randomly
        // sampled point around the pixel location i, j.

        auto offset = sample_square();
        auto pixel_sample = pixel00_loc
                          + ((i + offset.x()) * pixel_delta_u)
                          + ((j + offset.y()) * pixel_delta_v);

        auto ray_origin = (defocus_angle <= 0) ? center : defocus_disk_sample();
        auto ray_direction = pixel_sample - ray_origin;

        return ray(ray_origin, ray_direction);
    }

    vec3 sample_square() const {
        // Returns the vector to a random point in the [-.5,-.5]-[+.5,+.5] unit square.
        return vec3(random_double() - 0.5, random_double() - 0.5, 0);
    }

    vec3 sample_disk(double radius) const {
        // Returns a random point in the unit (radius 0.5) disk centered at the origin.
        return radius * random_in_unit_disk();
    }

    point3 defocus_disk_sample() const {
        // Returns a random point in the camera defocus disk.
        auto p = random_in_unit_disk();
        return center + (p[0] * defocus_disk_u) + (p[1] * defocus_disk_v);
    }

    color ray_color(const ray& r, int depth, const hittable& world) const {
        // If we've exceeded the ray bounce limit, no more light is gathered.
        if (depth <= 0)
            return color(0,0,0);

        hit_record rec;

        if (world.hit(r, interval(0.001, infinity), rec)) {
            ray scattered;
            color attenuation;
            if (rec.mat->scatter(r, rec, attenuation, scattered))
                return attenuation * ray_color(scattered, depth-1, world);
            return rec.mat->emitted();
        }

        vec3 unit_direction = unit_vector(r.direction());
        auto a = 0.5*(unit_direction.y() + 1.0);
        return (1.0-a)*color(1.0, 1.0, 1.0) + a*color(0.5, 0.7, 1.0);
    }

    // non-recursive, faster ray_color for space conditions with black background
    // this won't currently work for glass or materials with multiple paths
    color ray_color_space(const ray& r, int depth, const hittable& world) const {
        color throughput(1, 1, 1);
        ray current_ray = r;

        for (int bounce = 0; bounce < depth; ++bounce) {
            hit_record world_rec;
            const bool hit_world = world.hit(current_ray, interval(0.001, infinity), world_rec);
            const double t_world = hit_world ? world_rec.t : infinity;

            double t_light = infinity;
            if (hit_light(current_ray, t_light) && t_light < t_world) {
                return throughput * light_color;
            }

            if (!hit_world) {
                // Space miss: black background by spec.
                return color(0,0,0);
            }

            ray scattered;
            color attenuation;
            if (!world_rec.mat->scatter(current_ray, world_rec, attenuation, scattered)) {
                return throughput * world_rec.mat->emitted();
            }

            throughput = throughput * attenuation;
            current_ray = scattered;
        }
        return color(0,0,0);
    }

    bool hit_light(const ray& r, double& t_light) const {
        vec3 oc = r.origin() - light_center;
        double a = dot(r.direction(), r.direction());
        double half_b = dot(oc, r.direction());
        double c = dot(oc, oc) - light_radius * light_radius;
        double discriminant = half_b * half_b - a * c;
        if (discriminant < 0) {
            return false;
        }

        double sqrtd = std::sqrt(discriminant);
        double root = (-half_b - sqrtd) / a;
        if (root <= 0.001) {
            root = (-half_b + sqrtd) / a;
            if (root <= 0.001) {
                return false;
            }
        }

        t_light = root;
        return true;
    }

};



Writing /content/raytrace_submission/camera.h


## Write `main.cc`

This snippet writes the main driver. It reads the config, builds the scene, supports Bézier motion, quaternion orientation smoothing, subject-locked orbit shots, and streams frames to FFmpeg for MP4 output.

In [15]:
%%writefile /content/raytrace_submission/main.cc
//==============================================================================================
// Originally written in 2016 by Peter Shirley <ptrshrl@gmail.com>
//
// To the extent possible under law, the author(s) have dedicated all copyright and related and
// neighboring rights to this software to the public domain worldwide. This software is
// distributed without any warranty.
//
// You should have received a copy (see file COPYING.txt) of the CC0 Public Domain Dedication
// along with this software. If not, see <http://creativecommons.org/publicdomain/zero/1.0/>.
//==============================================================================================
#include "rtweekend.h"
#include <string>
#include <fstream>
#include <sstream>
#include <iomanip>
#include <cstdio>
#include <cstring>
#include <map>
#include <vector>
#include "camera.h"
#include "hittable.h"
#include "hittable_list.h"
#include "bvh.h"
#include "material.h"
#include "sphere.h"
#include <webp/encode.h>
using namespace std;

double g_anim_time_seconds = 0.0;
enum class CameraMotionMode {
    Linear,
    Bezier,
    OrbitLocked
};
enum class TimeWarpMode {
    Linear,
    Smoothstep,
    Smootherstep
};
enum class OrientationMode {
    ConfigPath,
    SlerpForward
};
struct quaternion {
    double w, x, y, z;
    quaternion(double w_=1.0, double x_=0.0, double y_=0.0, double z_=0.0)
        : w(w_), x(x_), y(y_), z(z_) {}
    static quaternion from_axis_angle(const vec3& axis, double angle_radians) {
        vec3 a = unit_vector(axis);
        const double half = 0.5 * angle_radians;
        const double s = std::sin(half);
        return quaternion(std::cos(half), a.x() * s, a.y() * s, a.z() * s);
    }
    static quaternion from_two_vectors(const vec3& from, const vec3& to) {
        vec3 f = unit_vector(from);
        vec3 t = unit_vector(to);
        const double cos_theta = dot(f, t);
        if (cos_theta < -0.999999) {
            vec3 axis = cross(vec3(1, 0, 0), f);
            if (axis.length_squared() < 1e-12) {
                axis = cross(vec3(0, 1, 0), f);
            }
            return from_axis_angle(axis, pi);
        }
        vec3 axis = cross(f, t);
        const double s = std::sqrt((1.0 + cos_theta) * 2.0);
        const double invs = 1.0 / s;
        return quaternion(0.5 * s, axis.x() * invs, axis.y() * invs, axis.z() * invs).normalized();
    }
    double norm() const {
        return std::sqrt(w*w + x*x + y*y + z*z);
    }
    quaternion normalized() const {
        const double n = norm();
        if (n <= 1e-12) return quaternion();
        return quaternion(w / n, x / n, y / n, z / n);
    }
    quaternion conjugate() const {
        return quaternion(w, -x, -y, -z);
    }
    quaternion operator*(const quaternion& q) const {
        return quaternion(
            w*q.w - x*q.x - y*q.y - z*q.z,
            w*q.x + x*q.w + y*q.z - z*q.y,
            w*q.y - x*q.z + y*q.w + z*q.x,
            w*q.z + x*q.y - y*q.x + z*q.w
        );
    }
    quaternion operator*(double s) const {
        return quaternion(w * s, x * s, y * s, z * s);
    }
    quaternion operator+(const quaternion& q) const {
        return quaternion(w + q.w, x + q.x, y + q.y, z + q.z);
    }
    static quaternion slerp(quaternion q0, quaternion q1, double t) {
        q0 = q0.normalized();
        q1 = q1.normalized();
        double cos_theta = q0.w*q1.w + q0.x*q1.x + q0.y*q1.y + q0.z*q1.z;
        if (cos_theta < 0.0) {
            q1 = q1 * -1.0;
            cos_theta = -cos_theta;
        }
        if (cos_theta > 0.9995) {
            return (q0 * (1.0 - t) + q1 * t).normalized();
        }
        const double theta = std::acos(std::max(-1.0, std::min(1.0, cos_theta)));
        const double sin_theta = std::sin(theta);
        const double w0 = std::sin((1.0 - t) * theta) / sin_theta;
        const double w1 = std::sin(t * theta) / sin_theta;
        return (q0 * w0 + q1 * w1).normalized();
    }
};
vec3 rotate_by_quaternion(const vec3& v, const quaternion& q) {
    quaternion p(0.0, v.x(), v.y(), v.z());
    quaternion r = q * p * q.conjugate();
    return vec3(r.x, r.y, r.z);
}
double apply_time_warp(TimeWarpMode mode, double t) {
    if (t < 0.0) t = 0.0;
    if (t > 1.0) t = 1.0;
    switch (mode) {
        case TimeWarpMode::Smoothstep:
            return t * t * (3.0 - 2.0 * t);
        case TimeWarpMode::Smootherstep: {
            // Apply smoothstep twice to get a stronger ease-in / ease-out curve.
            double s = t * t * (3.0 - 2.0 * t);
            return s * s * (3.0 - 2.0 * s);
        }
        case TimeWarpMode::Linear:
        default:
            return t;
    }
}
vec3 cubic_bezier(const vec3& p0, const vec3& p1, const vec3& p2, const vec3& p3, double t) {
    const double u = 1.0 - t;
    return (u*u*u) * p0
         + (3.0 * u*u * t) * p1
         + (3.0 * u * t*t) * p2
         + (t*t*t) * p3;
}
map<string, vector<string>> read_remaining_config(ifstream& conf) {
    map<string, vector<string>> extras;
    string line;
    while (getline(conf, line)) {
        if (line.empty()) continue;
        stringstream ss(line);
        string key;
        ss >> key;
        if (key.empty()) continue;
        string value;
        vector<string> values;
        while (ss >> value) {
            values.push_back(value);
        }
        extras[key] = values;
    }
    return extras;
}
string read_extra_string(const map<string, vector<string>>& extras, const string& key, const string& default_value) {
    auto it = extras.find(key);
    if (it == extras.end() || it->second.empty()) return default_value;
    return it->second[0];
}
double read_extra_double(const map<string, vector<string>>& extras, const string& key, double default_value) {
    auto it = extras.find(key);
    if (it == extras.end() || it->second.empty()) return default_value;
    return stod(it->second[0]);
}
vec3 read_extra_vec3(const map<string, vector<string>>& extras, const string& key, vec3 default_value) {
    auto it = extras.find(key);
    if (it == extras.end() || it->second.size() < 3) return default_value;
    return vec3(stod(it->second[0]), stod(it->second[1]), stod(it->second[2]));
}
string read_config(ifstream &conf, const string &key, string default_value) {
    string line;
    getline(conf, line);
    stringstream ss(line);
    string confkey, value;
    ss >> confkey >> value;
    if (key != confkey)
        cerr << "Key " << key << " not found in config file\n";
    return (key == confkey) ? value : default_value;
}
int read_config(ifstream &conf, const string &key, int default_value) {
    string line;
    getline(conf, line);
    stringstream ss(line);
    string confkey, value;
    ss >> confkey >> value;
    if (key != confkey)
        cerr << "Key " << key << " not found in config file\n";
    return (key == confkey) ? stoi(value) : default_value;
}
double read_config(ifstream &conf, const string &key, double default_value) {
    string line;
    getline(conf, line);
    stringstream ss(line);
    string confkey, value;
    ss >> confkey >> value;
    if (key != confkey)
        cerr << "Key " << key << " not found in config file\n";
    return (key == confkey) ? stod(value) : default_value;
}
vec3 read_config(ifstream &conf, const string &key, vec3 default_value) {
    string line;
    getline(conf, line);
    stringstream ss(line);
    string confkey;
    double x, y, z;
    ss >> confkey;
    if (key != confkey) {
        cerr << "Key " << key << " not found in config file\n";
        return default_value;
    }
    if (ss >> x >> y >> z)
        return vec3(x, y, z);
    cerr << "Could not read 3 values for " << key << "\n";
    return default_value;
}

inline double clamp01(double x) {
    if (x < 0.0) return 0.0;
    if (x > 1.0) return 1.0;
    return x;
}

inline double smoother01(double t) {
    t = clamp01(t);
    const double s = t * t * (3.0 - 2.0 * t);
    return s * s * (3.0 - 2.0 * s);
}

inline
double phase01(double t, double t0, double t1) {
    if (t <= t0) return 0.0;
    if (t >= t1) return 1.0;
    const double x = (t - t0) / (t1 - t0);
    return x * x * (3.0 - 2.0 * x);
}

point3 point_lerp(const point3& a, const point3& b, double t) {
    return a + t * (b - a);
}

struct billiard_ball_state {
    point3 center;
    vec3 velocity;
    double radius;
    double mass;
};

double distance_to_segment_xz(const point3& p, const point3& a, const point3& b) {
    vec3 pa(p.x() - a.x(), 0.0, p.z() - a.z());
    vec3 ba(b.x() - a.x(), 0.0, b.z() - a.z());
    const double denom = ba.length_squared();
    if (denom <= 1e-12) return pa.length();
    const double h = std::max(0.0, std::min(1.0, dot(pa, ba) / denom));
    return (pa - h * ba).length();
}

void advance_billiard_step(std::vector<billiard_ball_state>& balls, double dt) {
    const double wall_restitution = 0.94;
    const double ball_restitution = 0.96;
    const double damping = 0.9992;

    const double xmin = -7.6;
    const double xmax =  7.6;
    const double zmin = -8.8;
    const double zmax =  5.8;

    for (auto& ball : balls) {
        ball.center += dt * ball.velocity;

        if (ball.center.x() - ball.radius < xmin && ball.velocity.x() < 0.0) {
            ball.center = point3(xmin + ball.radius, ball.center.y(), ball.center.z());
            ball.velocity = vec3(-ball.velocity.x() * wall_restitution, ball.velocity.y(), ball.velocity.z());
        }
        if (ball.center.x() + ball.radius > xmax && ball.velocity.x() > 0.0) {
            ball.center = point3(xmax - ball.radius, ball.center.y(), ball.center.z());
            ball.velocity = vec3(-ball.velocity.x() * wall_restitution, ball.velocity.y(), ball.velocity.z());
        }
        if (ball.center.z() - ball.radius < zmin && ball.velocity.z() < 0.0) {
            ball.center = point3(ball.center.x(), ball.center.y(), zmin + ball.radius);
            ball.velocity = vec3(ball.velocity.x(), ball.velocity.y(), -ball.velocity.z() * wall_restitution);
        }
        if (ball.center.z() + ball.radius > zmax && ball.velocity.z() > 0.0) {
            ball.center = point3(ball.center.x(), ball.center.y(), zmax - ball.radius);
            ball.velocity = vec3(ball.velocity.x(), ball.velocity.y(), -ball.velocity.z() * wall_restitution);
        }
    }

    for (int pass = 0; pass < 2; ++pass) {
        for (size_t i = 0; i < balls.size(); ++i) {
            for (size_t j = i + 1; j < balls.size(); ++j) {
                auto& a = balls[i];
                auto& b = balls[j];

                vec3 delta(b.center.x() - a.center.x(), 0.0, b.center.z() - a.center.z());
                const double dist_sq = delta.length_squared();
                const double min_dist = a.radius + b.radius;
                if (dist_sq > min_dist * min_dist) continue;

                const double dist = std::sqrt(std::max(1e-16, dist_sq));
                vec3 normal = (dist > 1e-8) ? (delta / dist) : vec3(1.0, 0.0, 0.0);
                const double penetration = min_dist - dist;

                const double inv_mass_a = 1.0 / a.mass;
                const double inv_mass_b = 1.0 / b.mass;
                const double inv_mass_sum = inv_mass_a + inv_mass_b;

                a.center = a.center - normal * (penetration * (inv_mass_a / inv_mass_sum));
                b.center = b.center + normal * (penetration * (inv_mass_b / inv_mass_sum));

                vec3 relative_velocity = b.velocity - a.velocity;
                const double rel_normal = dot(relative_velocity, normal);
                if (rel_normal < 0.0) {
                    const double impulse = -(1.0 + ball_restitution) * rel_normal / inv_mass_sum;
                    a.velocity = a.velocity - normal * (impulse * inv_mass_a);
                    b.velocity = b.velocity + normal * (impulse * inv_mass_b);
                }
            }
        }
    }

    for (auto& ball : balls) {
        ball.velocity = damping * ball.velocity;
        if (ball.velocity.length() < 0.003) {
            ball.velocity = vec3(0.0, 0.0, 0.0);
        }
    }
}

void simulate_repo_style_billiards(
    point3& center_pos,
    point3& left_pos,
    point3& right_pos,
    point3& front_pos,
    point3& cue_pos
) {
    const point3 center_base(0.0, 1.0, 0.0);
    const point3 left_base(-4.0, 1.0, 0.0);
    const point3 right_base(4.0, 1.0, 0.0);
    const point3 front_base(1.4, 0.55, -2.4);

    // Keep the exact repo-style hero balls, then let a cue ball break them over time.
    // This aim produces a deterministic chain: cue -> front accent -> right hero -> center glass -> left hero.
    const point3 break_target(3.0, 0.55, 0.0);
    vec3 cue_dir = unit_vector(vec3(
        break_target.x() - front_base.x(),
        0.0,
        break_target.z() - front_base.z()
    ));
    const point3 cue_start = front_base - 6.3 * cue_dir + vec3(0.0, 0.35 - front_base.y(), 0.0);

    std::vector<billiard_ball_state> balls;
    balls.push_back({center_base, vec3(0.0, 0.0, 0.0), 1.0, 1.0});
    balls.push_back({left_base,   vec3(0.0, 0.0, 0.0), 1.0, 1.0});
    balls.push_back({right_base,  vec3(0.0, 0.0, 0.0), 1.0, 1.0});
    balls.push_back({front_base,  vec3(0.0, 0.0, 0.0), 0.55, 0.55 * 0.55});
    balls.push_back({cue_start,   6.0 * cue_dir,       0.35, 0.35 * 0.35});

    const double dt = 1.0 / 240.0;
    const double total_seconds = std::max(0.0, g_anim_time_seconds);
    const int whole_steps = static_cast<int>(std::floor(total_seconds / dt));
    const double remainder = total_seconds - whole_steps * dt;

    for (int step = 0; step < whole_steps; ++step) {
        advance_billiard_step(balls, dt);
    }
    if (remainder > 1e-9) {
        advance_billiard_step(balls, remainder);
    }

    center_pos = balls[0].center;
    left_pos   = balls[1].center;
    right_pos  = balls[2].center;
    front_pos  = balls[3].center;
    cue_pos    = balls[4].center;
}

void build_simple_world(hittable_list &world, double anim_t = 0.0) {
    (void)anim_t;

    auto ground_material = make_shared<lambertian>(color(0.52, 0.52, 0.52));
    world.add(make_shared<sphere>(point3(0,-1000,0), 1000, ground_material));

    point3 center_pos, left_pos, right_pos, front_pos, cue_pos;
    simulate_repo_style_billiards(center_pos, left_pos, right_pos, front_pos, cue_pos);

    // Build a deterministic field so rebuilding the world each frame keeps the same sphere layout.
    std::mt19937_64 rng(5602026ULL);
    auto rand01 = [&]() {
        static std::uniform_real_distribution<double> dist01(0.0, 1.0);
        return dist01(rng);
    };
    auto rand_range = [&](double lo, double hi) {
        std::uniform_real_distribution<double> dist(lo, hi);
        return dist(rng);
    };
    auto rand_color = [&](double lo, double hi) {
        return color(rand_range(lo, hi), rand_range(lo, hi), rand_range(lo, hi));
    };

    const point3 break_target(3.0, 0.55, 0.0);
    const point3 front_base(1.4, 0.55, -2.4);

    const int field_min = -5;
    const int field_max = 6;

    for (int a = field_min; a < field_max; ++a) {
        for (int b = field_min; b < field_max; ++b) {
            auto choose_mat = rand01();
            point3 center(a + 0.9 * rand01(), 0.20, b + 0.9 * rand01());

            // Keep the action corridor open so the same repo-style hero balls can collide cleanly.
            if ((center - center_pos).length() <= 1.35) continue;
            if ((center - left_pos).length() <= 1.35) continue;
            if ((center - right_pos).length() <= 1.35) continue;
            if ((center - front_pos).length() <= 0.90) continue;
            if ((center - cue_pos).length() <= 0.70) continue;
            if (distance_to_segment_xz(center, cue_pos, front_base) <= 0.48) continue;
            if (distance_to_segment_xz(center, front_base, break_target) <= 0.45) continue;
            if (distance_to_segment_xz(center, point3(0.0, 0.2, 0.0), point3(-4.0, 0.2, 0.0)) <= 0.40) continue;

            shared_ptr<material> sphere_material;
            if (choose_mat < 0.74) {
                auto albedo = rand_color(0.18, 0.96) * rand_color(0.35, 1.0);
                sphere_material = make_shared<lambertian>(albedo);
            } else if (choose_mat < 0.92) {
                auto albedo = rand_color(0.55, 1.0);
                auto fuzz = rand_range(0.0, 0.18);
                sphere_material = make_shared<metal>(albedo, fuzz);
            } else {
                sphere_material = make_shared<dielectric>(1.5);
            }

            world.add(make_shared<sphere>(center, 0.20, sphere_material));
        }
    }

    auto center_glass = make_shared<dielectric>(1.5);
    world.add(make_shared<sphere>(center_pos, 1.0, center_glass));

    auto left_repo = make_shared<spinning_uv_lambertian>(
        color(0.42, 0.23, 0.12), color(0.78, 0.63, 0.41), 0.06, 10.0, 4.0, 0.06);
    world.add(make_shared<sphere>(left_pos, 1.0, left_repo));

    auto right_metal = make_shared<metal>(color(0.78, 0.79, 0.82), 0.02);
    world.add(make_shared<sphere>(right_pos, 1.0, right_metal));

    auto front_accent = make_shared<spinning_uv_lambertian>(
        color(0.12, 0.44, 0.90), color(0.96, 0.90, 0.16), 0.10, 12.0, 6.0, 0.08);
    world.add(make_shared<sphere>(front_pos, 0.55, front_accent));

    auto cue_material = make_shared<lambertian>(color(0.92, 0.92, 0.88));
    world.add(make_shared<sphere>(cue_pos, 0.35, cue_material));

    cout << "Built world: " << world.objects.size() << " objects\n";
}
void build_world(hittable_list &world, int min_coord, int max_coord) {
    auto ground_material = make_shared<lambertian>(color(0.5, 0.5, 0.5));
    world.add(make_shared<sphere>(point3(0,-1000,0), 1000, ground_material));
    for (int a = min_coord; a < max_coord; a++) {
        for (int b = min_coord; b < max_coord; b++) {
            auto choose_mat = random_double();
            point3 center(a + 0.9*random_double(), 0.2, b + 0.9*random_double());
            if ((center - point3(4, 0.2, 0)).length() > 0.9) {
                shared_ptr<material> sphere_material;
                if (choose_mat < 0.8) {
                    // diffuse
                    auto albedo = color::random() * color::random();
                    sphere_material = make_shared<lambertian>(albedo);
                    world.add(make_shared<sphere>(center, 0.2, sphere_material));
                } else if (choose_mat < 0.95) {
                    // metal
                    auto albedo = color::random(0.5, 1);
                    auto fuzz = random_double(0, 0.5);
                    sphere_material = make_shared<metal>(albedo, fuzz);
                    world.add(make_shared<sphere>(center, 0.2, sphere_material));
                } else {
                    // glass
                    sphere_material = make_shared<dielectric>(1.5);
                    world.add(make_shared<sphere>(center, 0.2, sphere_material));
                }
            }
        }
    }
    auto material1 = make_shared<dielectric>(1.5);
    world.add(make_shared<sphere>(point3(0, 1, 0), 1.0, material1));
    auto material2 = make_shared<lambertian>(color(0.4, 0.2, 0.1));
    world.add(make_shared<sphere>(point3(-4, 1, 0), 1.0, material2));
    auto material3 = make_shared<metal>(color(0.7, 0.6, 0.5), 0.0);
    world.add(make_shared<sphere>(point3(4, 1, 0), 1.0, material3));
    cout << "Built world: " << world.objects.size() << " objects\n";
}
void build_space_world(hittable_list &world) {
    // Preserve original mirror setup and add an earth-like sphere.
    auto red = make_shared<lambertian>(color(0.65, 0.05, 0.05));
    auto silver = make_shared<metal>(color(1.0, 1.0, 1.0), 0.0);
    world.add(make_shared<sphere>(point3(2.9, 3.2, 0), 0.8, red));
    world.add(make_shared<sphere>(point3(4,0,0), 1.3, silver));
    world.add(make_shared<sphere>(point3(0,0,0), 2.2, silver));
}
void build_space_world_with_light(hittable_list &world) {
    // the light is built into the config file
//    auto light = make_shared<diffuse_light>(color(1, 1, 1));
//    world.add(make_shared<sphere>(point3(0, 8, 0), 1.0, light));
    auto earth = make_shared<lambertian>(color(0.12, 0.27, 0.70));
    world.add(make_shared<sphere>(point3(18, 0, 0.0), 1.5, earth));
    auto moon = make_shared<lambertian>(color(0.55, 0.56, 0.60));
    world.add(make_shared<sphere>(point3(14, 0, 0.4), 0.7, moon));
}
FILE* setup_video_pipe(int width, int height, int fps, const char* filename, const string& filter, const string& preset, int crf) {
    char cmd[1024];
    if (!filter.empty()) {
        snprintf(cmd, sizeof(cmd),
            "ffmpeg -y -f rawvideo -pix_fmt rgba -s %dx%d -r %d -i - -vf %s "
            "-c:v libx264 -preset %s -crf %d -pix_fmt yuv420p -movflags +faststart %s",
            width, height, fps, filter.c_str(), preset.c_str(), crf, filename);
    } else {
        snprintf(cmd, sizeof(cmd),
            "ffmpeg -y -f rawvideo -pix_fmt rgba -s %dx%d -r %d -i - "
            "-c:v libx264 -preset %s -crf %d -pix_fmt yuv420p -movflags +faststart %s",
            width, height, fps, preset.c_str(), crf, filename);
    }
    return popen(cmd, "w");
}
void write_frame(FILE* pipe, const uint8_t* rgba_data, size_t frame_size) {
    fwrite(rgba_data, 1, frame_size, pipe);
    fflush(pipe);
}
void close_video_pipe(FILE* pipe) {
    pclose(pipe);
}
void write_webp(const char filename[], uint32_t* pixels, int width, int height) {
    uint8_t* output;
    size_t output_size;
    output_size = WebPEncodeRGBA((uint8_t*)pixels, width, height,
                    width * 4, 90.0f, &output);
    FILE* fp = fopen(filename, "wb");
    if (fp) {
        fwrite(output, output_size, 1, fp);
        fclose(fp);
    }
    WebPFree(output);
}
void render(const char conffile[]) {
    ifstream conf(conffile);
    string filename = read_config(conf, "filename", "test");
    uint32_t w = read_config(conf, "w", 1200);
    uint32_t h = read_config(conf, "h", 1024);
    uint32_t num_frames = read_config(conf, "num_frames", 1);
    uint32_t worldid = read_config(conf, "world", 0);
    int min_coord = read_config(conf, "min_coord", -11);
    int max_coord = read_config(conf, "max_coord", 11);

    // Store camera settings locally
    int samples_per_pixel = read_config(conf, "samples_per_pixel", 10);
    int max_depth = read_config(conf, "max_depth", 20);
    double vfov = read_config(conf, "vfov", 20.0);
    double defocus_angle = read_config(conf, "defocus_angle", 0.1);
    double focus_dist = read_config(conf, "focus_dist", 10.0);
    vec3 lookfrom0 = read_config(conf, "lookfrom0", vec3(13,2,3));
    vec3 lookat0 = read_config(conf, "lookat0", vec3(0,0,0));
    vec3 vup0 = read_config(conf, "vup0", vec3(0,1,0));
    vec3 lookfrom1 = read_config(conf, "lookfrom1", vec3(13,2,3));
    vec3 lookat1 = read_config(conf, "lookat1", vec3(0,0,0));
    vec3 vup1 = read_config(conf, "vup1", vec3(0,1,0));
    vec3 light_center = vec3(0, 8, 0);
    double light_radius = 1.0;
    vec3 light_color = vec3(1, 1, 1);
    if (worldid == 3) {
        // These lines are optional in config, but should be present for deterministic setup.
        light_center = read_config(conf, "light_center", vec3(0, 8, 0));
        light_radius = read_config(conf, "light_radius", 1.0);
        light_color = read_config(conf, "light_color", vec3(1, 1, 1));
    }
    const map<string, vector<string>> extras = read_remaining_config(conf);
    const string motion_mode_str = read_extra_string(extras, "camera_motion", "linear");
    CameraMotionMode motion_mode = CameraMotionMode::Linear;
    if (motion_mode_str == "bezier") {
        motion_mode = CameraMotionMode::Bezier;
    } else if (motion_mode_str == "orbit_locked") {
        motion_mode = CameraMotionMode::OrbitLocked;
    }
    const string time_warp_str = read_extra_string(extras, "time_warp", "linear");
    TimeWarpMode time_warp = TimeWarpMode::Linear;
    if (time_warp_str == "smoothstep") {
        time_warp = TimeWarpMode::Smoothstep;
    } else if (time_warp_str == "smootherstep") {
        time_warp = TimeWarpMode::Smootherstep;
    }
    const string orientation_mode_str = read_extra_string(extras, "orientation_mode", "config_path");
    const OrientationMode orientation_mode = (orientation_mode_str == "slerp_forward")
        ? OrientationMode::SlerpForward
        : OrientationMode::ConfigPath;
    const int fps = static_cast<int>(read_extra_double(extras, "fps", 30.0));
    const int render_threads = static_cast<int>(read_extra_double(extras, "render_threads", 0.0));
    const int requested_bvh = static_cast<int>(read_extra_double(extras, "use_bvh", 1.0));
    const int video_crf = static_cast<int>(read_extra_double(extras, "video_crf", 18.0));
    const string video_preset = read_extra_string(extras, "video_preset", "medium");
    const string video_filter = read_extra_string(extras, "video_filter", "");
    const double roll_start_deg = read_extra_double(extras, "roll_start_deg", 0.0);
    const double roll_end_deg = read_extra_double(extras, "roll_end_deg", 0.0);
    const vec3 lookfrom_cp1 = read_extra_vec3(extras, "lookfrom_cp1",
        read_extra_vec3(extras, "lookfrom2", lookfrom0 + vec3(0.0, 2.5, 0.0)));
    const vec3 lookfrom_cp2 = read_extra_vec3(extras, "lookfrom_cp2",
        read_extra_vec3(extras, "lookfrom3", lookfrom1 + vec3(0.0, 2.5, 0.0)));
    const vec3 lookat_cp1 = read_extra_vec3(extras, "lookat_cp1",
        read_extra_vec3(extras, "lookat2", lookat0));
    const vec3 lookat_cp2 = read_extra_vec3(extras, "lookat_cp2",
        read_extra_vec3(extras, "lookat3", lookat1));
    const vec3 vup_cp1 = read_extra_vec3(extras, "vup_cp1",
        read_extra_vec3(extras, "vup2", vup0));
    const vec3 vup_cp2 = read_extra_vec3(extras, "vup_cp2",
        read_extra_vec3(extras, "vup3", vup1));
    const vec3 orbit_center = read_extra_vec3(extras, "orbit_center", lookat0);
    const double orbit_radius_default = std::sqrt((lookfrom0.x() - orbit_center.x()) * (lookfrom0.x() - orbit_center.x())
                                             + (lookfrom0.z() - orbit_center.z()) * (lookfrom0.z() - orbit_center.z()));
    const double orbit_start_default = 180.0 * std::atan2(lookfrom0.z() - orbit_center.z(),
                                                           lookfrom0.x() - orbit_center.x()) / pi;
    const double orbit_end_default = orbit_start_default + 360.0;
    const double orbit_radius = read_extra_double(extras, "orbit_radius", orbit_radius_default > 1e-9 ? orbit_radius_default : 8.0);
    const double orbit_start_deg = read_extra_double(extras, "orbit_start_deg", orbit_start_default);
    const double orbit_end_deg = read_extra_double(extras, "orbit_end_deg", orbit_end_default);
    const double orbit_height_start = read_extra_double(extras, "orbit_height_start", lookfrom0.y());
    const double orbit_height_end = read_extra_double(extras, "orbit_height_end", lookfrom1.y());
    const vec3 orbit_lookat = read_extra_vec3(extras, "orbit_lookat", lookat0);
    const vec3 orbit_vup = read_extra_vec3(extras, "orbit_vup", vup0);
    hittable_list world;
    switch (worldid) {
        case 0:
            build_simple_world(world, 0.0);
            break;
        case 1:
            build_world(world, min_coord, max_coord);
            break;
        case 2:
            build_space_world(world);
            break;
        case 3:
            build_space_world_with_light(world);
            break;
        default:
            cerr << "Invalid world number\n";
            return;
    }
    shared_ptr<hittable> render_world = make_shared<hittable_list>(world);
    if (requested_bvh && !world.objects.empty()) {
        render_world = make_shared<bvh_node>(world.objects);
    }
    camera cam;
    cam.aspect_ratio = 16.0 / 9.0;
    cam.image_width = w;
    cam.image_height = h;
    cam.samples_per_pixel = samples_per_pixel;
    cam.max_depth = max_depth;
    cam.vfov = vfov;
    cam.lookfrom = lookfrom0;
    cam.lookat = lookat0;
    cam.vup = vup0;
    cam.defocus_angle = defocus_angle;
    cam.focus_dist = focus_dist;
    cam.use_space_shader = (worldid == 3);
    cam.light_center = light_center;
    cam.light_radius = light_radius;
    cam.light_color = light_color;
    cam.render_threads = render_threads;
    cout << left
         << setw(20) << "filename:" << filename << '\n'
         << setw(20) << "w:" << w << '\n'
         << setw(20) << "h:" << h << '\n'
         << setw(20) << "world:" << worldid << '\n'
         << setw(20) << "min_coord:" << min_coord << '\n'
         << setw(20) << "max_coord:" << max_coord << '\n'
         << setw(20) << "samples_per_pixel:" << samples_per_pixel << '\n'
         << setw(20) << "max_depth:" << max_depth << '\n'
         << setw(20) << "vfov:" << vfov << '\n'
         << setw(20) << "defocus_angle:" << defocus_angle << '\n'
         << setw(20) << "focus_dist:" << focus_dist << '\n'
         << setw(20) << "lookfrom0:" << lookfrom0 << '\n'
         << setw(20) << "lookat0:" << lookat0 << '\n'
         << setw(20) << "vup0:" << vup0 << '\n'
         << setw(20) << "lookfrom1:" << lookfrom1 << '\n'
         << setw(20) << "lookat1:" << lookat1 << '\n'
         << setw(20) << "vup1:" << vup1 << '\n'
         << setw(20) << "camera_motion:" << motion_mode_str << '\n'
         << setw(20) << "orientation_mode:" << orientation_mode_str << '\n'
         << setw(20) << "fps:" << fps << '\n'
         << setw(20) << "render_threads:" << (render_threads > 0 ? render_threads : 0) << '\n'
         << setw(20) << "use_bvh:" << requested_bvh << '\n'
         << setw(20) << "video_crf:" << video_crf << '\n'
         << setw(20) << "video_preset:" << video_preset << '\n'
         << setw(20) << "video_filter:" << (video_filter.empty() ? string("(none)") : video_filter) << '\n';
    if (motion_mode == CameraMotionMode::Bezier) {
        cout << setw(20) << "lookfrom_cp1:" << lookfrom_cp1 << '\n'
             << setw(20) << "lookfrom_cp2:" << lookfrom_cp2 << '\n'
             << setw(20) << "lookat_cp1:" << lookat_cp1 << '\n'
             << setw(20) << "lookat_cp2:" << lookat_cp2 << '\n'
             << setw(20) << "vup_cp1:" << vup_cp1 << '\n'
             << setw(20) << "vup_cp2:" << vup_cp2 << '\n';
    }
    if (worldid == 3) {
        cout << setw(20) << "light_center:" << light_center << '\n'
             << setw(20) << "light_radius:" << light_radius << '\n'
             << setw(20) << "light_color:" << light_color << '\n';
    }
    const double per_frame = (num_frames > 1) ? (1.0 / double(num_frames - 1)) : 0.0;
    const double duration_seconds = (fps > 0) ? (static_cast<double>(num_frames) / static_cast<double>(fps)) : static_cast<double>(num_frames);
    const vec3 delta_lookfrom = lookfrom1 - lookfrom0;
    const vec3 delta_lookat = lookat1 - lookat0;
    const vec3 delta_vup = vup1 - vup0;
    const vec3 canonical_forward(0, 0, -1);
    const vec3 canonical_up(0, 1, 0);
    const vec3 forward0 = unit_vector(lookat0 - lookfrom0);
    const vec3 forward1 = unit_vector(lookat1 - lookfrom1);
    const double target_dist0 = (lookat0 - lookfrom0).length();
    const double target_dist1 = (lookat1 - lookfrom1).length();
    const quaternion orient_q0 = quaternion::from_two_vectors(canonical_forward, forward0);
    const quaternion orient_q1 = quaternion::from_two_vectors(canonical_forward, forward1);
    string video_filename = filename + ".mp4";
    FILE* pipe = num_frames > 1 ? setup_video_pipe(cam.image_width, cam.image_height, fps, video_filename.c_str(), video_filter, video_preset, video_crf) : nullptr;
    const size_t frame_size = cam.image_width * cam.image_height * 4;
    uint32_t* rgba_buffer = new uint32_t[cam.image_width * cam.image_height];

    // Render one frame at a time while keeping the motion logic explicit.
    for (int frame = 0; frame < num_frames; frame++) {
        const double t_raw = frame * per_frame;
        const double t = apply_time_warp(time_warp, t_raw);

        // METHOD A: move the camera body along a cubic Bézier path.
        if (motion_mode == CameraMotionMode::Bezier) {
            cam.lookfrom = cubic_bezier(lookfrom0, lookfrom_cp1, lookfrom_cp2, lookfrom1, t);
            cam.lookat = cubic_bezier(lookat0, lookat_cp1, lookat_cp2, lookat1, t);
            cam.vup = cubic_bezier(vup0, vup_cp1, vup_cp2, vup1, t);
        } else if (motion_mode == CameraMotionMode::OrbitLocked) {
            const double angle_deg = orbit_start_deg + t * (orbit_end_deg - orbit_start_deg);
            const double angle_rad = degrees_to_radians(angle_deg);
            const double orbit_height = orbit_height_start + t * (orbit_height_end - orbit_height_start);
            cam.lookfrom = orbit_center + vec3(orbit_radius * std::cos(angle_rad), orbit_height, orbit_radius * std::sin(angle_rad));
            cam.lookat = orbit_lookat;
            cam.vup = orbit_vup;
        } else {
            cam.lookfrom = lookfrom0 + t * delta_lookfrom;
            cam.lookat = lookat0 + t * delta_lookat;
            cam.vup = vup0 + t * delta_vup;
        }

        // METHOD B: keep the camera position and rotate the viewing direction with quaternion slerp.
        if (orientation_mode == OrientationMode::SlerpForward) {
            quaternion q = quaternion::slerp(orient_q0, orient_q1, t);
            vec3 forward_dir = rotate_by_quaternion(canonical_forward, q);
            vec3 up_dir = rotate_by_quaternion(canonical_up, q);
            const double target_dist = (1.0 - t) * target_dist0 + t * target_dist1;
            cam.lookat = cam.lookfrom + target_dist * forward_dir;
            cam.vup = up_dir;
        }

        const double roll_deg = roll_start_deg + t * (roll_end_deg - roll_start_deg);
        const vec3 forward_axis = cam.lookat - cam.lookfrom;
        if (forward_axis.length_squared() > 1e-12 && std::abs(roll_deg) > 1e-9) {
            quaternion roll_q = quaternion::from_axis_angle(forward_axis, degrees_to_radians(roll_deg));
            cam.vup = rotate_by_quaternion(cam.vup, roll_q);
        }

        // Drive the UV-based sphere spin from the full shot duration.
        g_anim_time_seconds = t * duration_seconds;

        shared_ptr<hittable> frame_world = render_world;
        if (worldid == 0) {
            hittable_list animated_world;
            build_simple_world(animated_world, t);
            frame_world = make_shared<hittable_list>(animated_world);
            if (requested_bvh && !animated_world.objects.empty()) {
                frame_world = make_shared<bvh_node>(animated_world.objects);
            }
        }

        cam.render(*frame_world, rgba_buffer, filename, frame);
        if (frame == 0) {
            write_webp((filename + ".webp").c_str(), rgba_buffer, cam.image_width, cam.image_height);
        }
        if (num_frames > 1) {
            write_frame(pipe, (uint8_t*)rgba_buffer, frame_size);
        }
    }
    if (num_frames > 1) {
        close_video_pipe(pipe);
    }
    delete[] rgba_buffer;
}
int main(int argc, char *argv[]) {
    if (argc < 2) {
        render("raytrace.conf");
        return 0;
    }
    for (int i = 1; i < argc; i++) {
        render(argv[i]);
    }
    return 0;
}



Writing /content/raytrace_submission/main.cc


## Write `Makefile`

This snippet writes the Makefile. It enables optimization, OpenMP, native vector instructions, and fast math so the Colab runtime can use the available CPU cores effectively.

In [16]:
%%writefile /content/raytrace_submission/Makefile
CXX = g++
CXXFLAGS = -std=c++11 -O3 -fopenmp -march=native -ffast-math

raytrace: main.cc camera.h hittable.h hittable_list.h material.h sphere.h rtweekend.h color.h interval.h ray.h vec3.h aabb.h bvh.h
	$(CXX) $(CXXFLAGS) -o raytrace main.cc -lwebp

clean:
	rm -f raytrace *.mp4 *.webp *.ppm


Writing /content/raytrace_submission/Makefile


## 3. Generate the configuration files inside the notebook

These configs target the updated repo-style sphere world while keeping the latest motion logic and fixes.

## Write the configuration files

This section writes the comparison and production configs. The production presets keep `use_bvh 1`, `time_warp smootherstep`, the denoise filter, and 10-second timing so the final movie remains stable and does not change the underlying camera choreography.

In [17]:

from pathlib import Path

def write_conf(path: Path, pairs):
    lines = [f"{key} {value}" for key, value in pairs]
    text = "\n".join(lines) + "\n"
    path.write_text(text, encoding="utf-8")
    return path

CONFIG_BASENAME_MAP = {}

# Quick Bézier test: curved flight around the repo-style sphere field.
motion_test_bezier_pairs = [
    ("filename", "motion_test_bezier"),
    ("w", "960"),
    ("h", "540"),
    ("num_frames", "180"),
    ("world", "0"),
    ("min_coord", "-5"),
    ("max_coord", "6"),
    ("samples_per_pixel", "24"),
    ("max_depth", "20"),
    ("vfov", "24"),
    ("defocus_angle", "0.18"),
    ("focus_dist", "12.0"),
    ("lookfrom0", "13 2.4 3"),
    ("lookfrom_cp1", "9 2.8 10"),
    ("lookfrom_cp2", "-9 2.8 10"),
    ("lookfrom1", "-13 2.4 3"),
    ("lookat0", "0 1 0"),
    ("lookat_cp1", "0 1 0"),
    ("lookat_cp2", "0 1 0"),
    ("lookat1", "0 1 0"),
    ("vup0", "0 1 0"),
    ("vup_cp1", "0 1 0"),
    ("vup_cp2", "0 1 0"),
    ("vup1", "0 1 0"),
    ("camera_motion", "bezier"),
    ("time_warp", "smootherstep"),
    ("orientation_mode", "config_path"),
    ("render_threads", "12"),
    ("use_bvh", "1"),
    ("fps", "30"),
    ("video_crf", "18"),
    ("video_preset", "medium"),
    ("video_filter", "hqdn3d=1.5:1.5:6:6"),
]

# Quick quaternion test: fixed camera position with a smooth tripod-style sweep.
motion_test_quaternion_pairs = [
    ("filename", "motion_test_quaternion"),
    ("w", "960"),
    ("h", "540"),
    ("num_frames", "180"),
    ("world", "0"),
    ("min_coord", "-5"),
    ("max_coord", "6"),
    ("samples_per_pixel", "24"),
    ("max_depth", "20"),
    ("vfov", "24"),
    ("defocus_angle", "0.0"),
    ("focus_dist", "12.0"),
    ("lookfrom0", "13 2.4 3"),
    ("lookat0", "-2.5 1.0 2.0"),
    ("vup0", "0 1 0"),
    ("lookfrom1", "13 2.4 3"),
    ("lookat1", "2.5 1.0 -2.0"),
    ("vup1", "0 1 0"),
    ("camera_motion", "linear"),
    ("time_warp", "smootherstep"),
    ("orientation_mode", "slerp_forward"),
    ("render_threads", "12"),
    ("use_bvh", "1"),
    ("fps", "30"),
    ("video_crf", "18"),
    ("video_preset", "medium"),
    ("video_filter", "hqdn3d=1.5:1.5:6:6"),
]

# Mid-cost cinematic orbit with the repo-style world.
cinematic_repo_spin_locked_pairs = [
    ("filename", "cinematic_repo_spin_locked"),
    ("w", "1280"),
    ("h", "720"),
    ("num_frames", "300"),
    ("world", "0"),
    ("min_coord", "-5"),
    ("max_coord", "6"),
    ("samples_per_pixel", "200"),
    ("max_depth", "40"),
    ("vfov", "24"),
    ("defocus_angle", "0.25"),
    ("focus_dist", "12.0"),
    ("lookfrom0", "13 2.4 3"),
    ("lookat0", "0 1 0"),
    ("vup0", "0 1 0"),
    ("lookfrom1", "13 2.4 3"),
    ("lookat1", "0 1 0"),
    ("vup1", "0 1 0"),
    ("camera_motion", "orbit_locked"),
    ("time_warp", "smootherstep"),
    ("orientation_mode", "config_path"),
    ("orbit_center", "0 0 0"),
    ("orbit_radius", "13.0"),
    ("orbit_start_deg", "12"),
    ("orbit_end_deg", "372"),
    ("orbit_height_start", "2.4"),
    ("orbit_height_end", "2.4"),
    ("orbit_lookat", "0 1 0"),
    ("orbit_vup", "0 1 0"),
    ("roll_start_deg", "0"),
    ("roll_end_deg", "0"),
    ("render_threads", "12"),
    ("use_bvh", "1"),
    ("fps", "30"),
    ("video_crf", "18"),
    ("video_preset", "medium"),
    ("video_filter", "hqdn3d=1.5:1.5:6:6"),
]

# Heavier final render using the same world, motion, and fixed naming logic.
cinematic_repo_spin_locked_hq_pairs = [
    ("filename", "cinematic_repo_spin_locked_hq"),
    ("w", "1280"),
    ("h", "720"),
    ("num_frames", "300"),
    ("world", "0"),
    ("min_coord", "-5"),
    ("max_coord", "6"),
    ("samples_per_pixel", "250"),
    ("max_depth", "50"),
    ("vfov", "24"),
    ("defocus_angle", "0.35"),
    ("focus_dist", "12.0"),
    ("lookfrom0", "13 2.4 3"),
    ("lookat0", "0 1 0"),
    ("vup0", "0 1 0"),
    ("lookfrom1", "13 2.4 3"),
    ("lookat1", "0 1 0"),
    ("vup1", "0 1 0"),
    ("camera_motion", "orbit_locked"),
    ("time_warp", "smootherstep"),
    ("orientation_mode", "config_path"),
    ("orbit_center", "0 0 0"),
    ("orbit_radius", "13.0"),
    ("orbit_start_deg", "12"),
    ("orbit_end_deg", "372"),
    ("orbit_height_start", "2.4"),
    ("orbit_height_end", "2.4"),
    ("orbit_lookat", "0 1 0"),
    ("orbit_vup", "0 1 0"),
    ("roll_start_deg", "0"),
    ("roll_end_deg", "0"),
    ("render_threads", "12"),
    ("use_bvh", "1"),
    ("fps", "30"),
    ("video_crf", "18"),
    ("video_preset", "medium"),
    ("video_filter", "hqdn3d=1.5:1.5:6:6"),
]

for conf_name, pairs in {
    "motion_test_bezier.conf": motion_test_bezier_pairs,
    "motion_test_quaternion.conf": motion_test_quaternion_pairs,
    "cinematic_repo_spin_locked.conf": cinematic_repo_spin_locked_pairs,
    "cinematic_repo_spin_locked_hq.conf": cinematic_repo_spin_locked_hq_pairs,
}.items():
    conf_path = write_conf(PROJECT_DIR / conf_name, pairs)
    filename_value = dict(pairs)["filename"]
    CONFIG_BASENAME_MAP[conf_path.name] = filename_value
    print(f"Wrote {conf_path.name} -> {filename_value}")


Wrote motion_test_bezier.conf -> motion_test_bezier
Wrote motion_test_quaternion.conf -> motion_test_quaternion
Wrote cinematic_repo_spin_locked.conf -> cinematic_repo_spin_locked
Wrote cinematic_repo_spin_locked_hq.conf -> cinematic_repo_spin_locked_hq


## 4. Build the renderer

This cell compiles the renderer from the source files written above. The Makefile enables OpenMP and native CPU optimization flags.


In [18]:
import subprocess

subprocess.run(["make", "clean"], cwd=PROJECT_DIR, check=False)
subprocess.run(["make"], cwd=PROJECT_DIR, check=True)
print("Build finished.")


Build finished.


## 5. Choose which configuration to run

Start with the quick Bézier preview. Once the repo-style balls look right, switch to the HQ orbit render.

In [19]:
CONFIG_TO_RUN = "motion_test_bezier.conf"
print("Selected config:", CONFIG_TO_RUN)
print()
print((PROJECT_DIR / CONFIG_TO_RUN).read_text(encoding="utf-8"))

Selected config: motion_test_bezier.conf

filename motion_test_bezier
w 960
h 540
num_frames 180
world 0
min_coord -5
max_coord 6
samples_per_pixel 24
max_depth 20
vfov 24
defocus_angle 0.18
focus_dist 12.0
lookfrom0 13 2.4 3
lookfrom_cp1 9 2.8 10
lookfrom_cp2 -9 2.8 10
lookfrom1 -13 2.4 3
lookat0 0 1 0
lookat_cp1 0 1 0
lookat_cp2 0 1 0
lookat1 0 1 0
vup0 0 1 0
vup_cp1 0 1 0
vup_cp2 0 1 0
vup1 0 1 0
camera_motion bezier
time_warp smootherstep
orientation_mode config_path
render_threads 12
use_bvh 1
fps 30
video_crf 18
video_preset medium
video_filter hqdn3d=1.5:1.5:6:6



## 6. Run the render

This cell launches the renderer on the selected configuration file. The renderer writes a first-frame WebP preview and, when there is more than one frame, an MP4 movie.


In [20]:
import subprocess

subprocess.run(["./raytrace", CONFIG_TO_RUN], cwd=PROJECT_DIR, check=True)
print("Render finished.")


Render finished.


## 7. Preview the output inside Colab

Colab does not embed WebP directly through `IPython.display.Image`, so this cell uses Pillow plus Matplotlib for the preview frame and the built-in video display for the MP4.


In [21]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt
from IPython.display import Video, display

output_name = CONFIG_BASENAME_MAP[CONFIG_TO_RUN]
OUTPUT_WEBP = PROJECT_DIR / f"{output_name}.webp"
OUTPUT_MP4 = PROJECT_DIR / f"{output_name}.mp4"

print("Preview image:", OUTPUT_WEBP)
print("Movie:", OUTPUT_MP4)

if OUTPUT_WEBP.exists():
    img = PILImage.open(OUTPUT_WEBP)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(output_name)
    plt.show()
else:
    print("Preview image not found.")

if OUTPUT_MP4.exists():
    display(Video(str(OUTPUT_MP4), embed=True))
else:
    print("Movie file not found.")


Output hidden; open in https://colab.research.google.com to view.

## 8. Download the output

Use this cell after the render completes if you want to save the MP4 or the WebP preview to your local machine.


In [22]:
from google.colab import files

# Uncomment one or both lines when the files exist.
files.download(str(OUTPUT_MP4))
files.download(str(OUTPUT_WEBP))


## 9. How to switch between the comparison shots

Try these configs in order:

1. `motion_test_bezier.conf` — curved spatial flight around the repo-style sphere world with the added billiard-style ball motion.
2. `motion_test_quaternion.conf` — tripod-style rotational sweep using quaternion slerp.
3. `cinematic_repo_spin_locked.conf` — longer orbit render with moderate production settings.
4. `cinematic_repo_spin_locked_hq.conf` — heavier final render.

To switch, change `CONFIG_TO_RUN` in section 5 and rerun sections 6–8.
